# Fase 3 · Memória de curto e longo prazo: o agente vira chatbot

**Projeto:** Assistente de Viagens com LangChain Agents
**Autora:** Shaiane Alves Tonin

O agente da Fase 2 raciocina e usa ferramentas, mas cada pergunta começa do zero. Nesta fase ele passa a **sustentar uma conversa**. Dentro da sessão, lembra do que foi dito (curto prazo). Entre sessões, lembra de quem é o viajante e do que já foi planejado (longo prazo).

| Requisito do enunciado | Seção |
|---|---|
| Curto vs. longo prazo no domínio, pela taxonomia CoALA | 1 |
| Gestão da janela de contexto (sliding window + sumarização + busca vetorial) | 2 |
| Comparação de ferramentas de memória (LangMem, Mem0, Letta/MemGPT) e decisão | 3 |
| Chatbot com `ConversationBufferMemory` | 4, 6 |
| Reconhecimento de intenção e extração de entidades | 5, 9 |
| Teste com consultas complexas de múltiplos turnos | 6, 7 |
| Memória de longo prazo entre sessões | 8 |

In [1]:
import os, sys, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.chdir("/content/projeto-langchain-agentes")
    %pip install -q -r requirements.txt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 140)

from viagens.config import USE_DUMMY, get_llm
from viagens.callbacks import LogExecucaoHandler
from viagens.chatbot import ChatbotViagens, AnaliseMensagem, INTENCOES
from viagens.memoria import MemoriaLongoPrazo, CHAVES_PERFIL

log_handler = LogExecucaoHandler(sessao="fase3")
USUARIO = "viajante_demo"
MemoriaLongoPrazo(USUARIO).apagar_tudo()   # começa do zero para a demonstração ser reprodutível
print("Modo dummy:", USE_DUMMY)

Modo dummy: False


## 1. Vocabulário e estratégia de memória (CoALA)

A taxonomia **CoALA** (*Cognitive Architectures for Language Agents*) separa a memória de **trabalho**, que é o que está no prompt agora, das memórias de **longo prazo**: semântica (fatos), episódica (experiências) e procedural (como fazer).

| Tipo | Horizonte | No Assistente de Viagens | Onde vive | Quando é escrita | Quando é lida |
|---|---|---|---|---|---|
| **Trabalho / curto prazo** | Sessão | Mensagens recentes, **estado do pedido** (destino, data, duração, viajantes, orçamento), resumo dos turnos antigos | `ConversationBufferMemory` + `estado` + `resumo_sessao` | A cada turno | A cada turno, inteira |
| **Semântica** | Permanente | Fatos estáveis do viajante: "vegetariana", "prefere voos diretos", "viaja com filho de 5 anos", "tem passaporte europeu" | `perfil.json` + índice FAISS dos fatos | Quando o NLU detecta uma preferência (ADD/UPDATE/NOOP) | Top-k por similaridade com a mensagem |
| **Episódica** | Permanente | O que aconteceu em cada sessão: "planejou Lisboa → trocou por Paris, março, 2 pessoas, R$ 15 mil" | Índice FAISS de resumos de sessão | Ao encerrar a sessão | Top-k por similaridade com a mensagem |
| **Procedural** | Permanente | *Como* atender: etapas de um plano, regras para ambiguidade, formato das respostas | Prompt PTCF, exemplos few-shot, política de diálogo no código | Pela engenharia (versionada) | Sempre, porque está no prompt |

**Critério de fronteira.** Um dado é de **curto prazo** se só vale para *esta* viagem ou conversa ("vamos em março", "somos 2"). É de **longo prazo** se continuaria verdadeiro numa viagem futura ("sou vegetariana"). Por isso o extrator de fatos (seção 4) é instruído a **não** gravar detalhes da viagem atual no perfil.

## 2. Gestão da janela de contexto

O prompt de cada chamada ao agente é montado em camadas, cada uma com um orçamento de tokens:

```
┌───────────────────────────────────────────────────────────────┐
│ PTCF + few-shot + ferramentas (procedural)   ~1.500 tokens, fixo │
├───────────────────────────────────────────────────────────────┤
│ Contexto do usuário                                 ~150–400 tokens │
│   • top-3 fatos do perfil      ← busca vetorial (FAISS)         │
│   • top-1 episódio anterior    ← busca vetorial (FAISS)         │
│   • estado do pedido (slots)                                   │
│   • resumo dos turnos antigos da sessão  ← sumarização          │
├───────────────────────────────────────────────────────────────┤
│ Histórico literal: ConversationBufferMemory  ≤ limite (900 tokens) │
│   → ao estourar, as mensagens mais antigas são resumidas e saem   │
│     do buffer; as 4 mais recentes ficam literais (sliding window) │
├───────────────────────────────────────────────────────────────┤
│ Pergunta + raciocínio ReAct (scratchpad)                         │
└───────────────────────────────────────────────────────────────┘
```

**Por que combinar as três técnicas:**
- **Sliding window** sozinha esquece o início da conversa, que é justamente quando o viajante diz destino, datas e restrições.
- **Sumarização** preserva esse começo de forma compacta, mas perde detalhes. Por isso os *slots* do pedido ficam num **estado estruturado** que nunca é resumido.
- **Busca vetorial** traz da memória de longo prazo só o que é **relevante para a mensagem atual**, em vez de despejar o perfil inteiro. O perfil cresce a cada viagem, e o custo do prompt não deve crescer junto.

**Atualização** (o *write path*): os fatos semânticos são atualizados no padrão ADD/UPDATE/NOOP. Uma mudança ("agora sou vegana") **substitui** o valor antigo, que vai para um histórico, em vez de criar um fato contraditório.

## 3. Ferramentas especializadas de memória: comparação e decisão

| | **LangMem** (LangChain) | **Mem0** | **Letta** (ex-MemGPT) |
|---|---|---|---|
| **O que é** | SDK de memória para agentes | Camada de memória ("memory layer"), open source e também serviço gerenciado | Framework/servidor para **agentes com estado** |
| **Arquitetura** | Ferramentas que o agente chama para gravar e buscar memórias, mais um *memory manager* em segundo plano que extrai e consolida memórias após a conversa. Armazena no `BaseStore` do LangGraph, com busca vetorial | A cada `add()`, um LLM extrai fatos e os compara com memórias semelhantes, decidindo **ADD/UPDATE/DELETE/NOOP**. Usa vector store (vários backends) e, opcionalmente, um grafo de conhecimento. As memórias têm escopo por `user_id`/`agent_id`/`run_id` | Inspirado em memória virtual de sistema operacional. A **core memory** (blocos editáveis) fica sempre no contexto; a **recall memory** (histórico pesquisável) e a **archival memory** (vetorial) ficam fora. O próprio agente move dados entre elas via *function calls* |
| **Tipos CoALA** | Semântica, episódica e **procedural** (otimiza o próprio prompt) | Semântica (fatos do usuário), com episódios em forma de fatos | Trabalho (core), episódica (recall) e semântica (archival) |
| **Caso de uso ideal** | Agentes LangGraph que precisam aprender com o uso | Personalização de qualquer app de LLM com poucas linhas | Agentes de longa duração que gerenciam a própria memória |
| **Custo de adoção aqui** | Exige LangGraph e o `BaseStore`; o agente deste projeto é um `AgentExecutor` | Mais um vector store e chamadas extras de LLM por mensagem; o modo gerenciado envia dados de usuário a terceiros | **Substitui** o executor: o agente passaria a rodar no servidor Letta, conflitando com o `AgentExecutor` exigido |

**Decisão: nenhuma entra como dependência nesta fase.** O agente usa uma implementação própria e leve (`viagens/memoria.py`) que **toma emprestados os dois melhores padrões**: o ciclo ADD/UPDATE/NOOP do Mem0, para não acumular fatos contraditórios, e a separação do Letta entre o que fica *sempre no contexto* (estado do pedido) e o que é *buscado sob demanda* (perfil e episódios).

Motivos:
1. O enunciado exige `AgentExecutor`/`initialize_agent` e `ConversationBufferMemory`. O Letta substituiria isso, e o LangMem pressupõe LangGraph.
2. A implementação própria reaproveita o FAISS e os embeddings que já estão no projeto e roda **offline no modo dummy**, requisito da entrega reprodutível.
3. Os dados do viajante (dieta, documentos) ficam locais e podem ser apagados (`apagar_tudo()`), sem depender de um serviço externo.

**Evolução prevista.** Na próxima disciplina, com a arquitetura multiagente em LangGraph, o **LangMem** é o candidato natural: o `perfil.json` e os episódios migram para o `BaseStore`, e o *memory manager* em segundo plano substitui o extrator síncrono de fatos.

## 4. A implementação

`viagens/chatbot.py` → `ChatbotViagens.turno(mensagem)`:

1. **NLU:** chain LCEL `prompt | llm | JsonOutputParser` → `AnaliseMensagem` (intenção, entidades, preferências e subtarefas). O prompt recebe o **estado** e o **histórico recente**, para resolver "lá" e "nessa época".
2. **Slot filling:** as entidades novas atualizam o estado. As mudanças ficam registradas.
3. **Memória semântica:** se houver preferências, uma chain extrai fatos `chave → valor` e grava no padrão ADD/UPDATE/NOOP.
4. **Política de diálogo:**
   - intenção de planejamento e dados obrigatórios faltando → **pergunta de coleta** (determinística, sem custo de LLM);
   - pedido completo → **plano** pelo agente ReAct, com o pedido reescrito em etapas, mais uma **verificação de cobertura**: se o agente responder sem executar alguma etapa obrigatória (regras, clima, câmbio), ele é chamado de novo com a lista do que falta;
   - só preferência → confirmação do que foi gravado;
   - pergunta sobre conversas anteriores (`consultar_memoria`) → agente instruído a usar a memória **episódica**;
   - demais casos → agente ReAct com `chat_history` + `contexto_usuario`.
5. **Curto prazo:** `ConversationBufferMemory.save_context` guarda a mensagem **original** do usuário (e não o pedido reescrito para o agente). Em seguida, a janela é gerenciada.

O agente ReAct é o mesmo da Fase 2 (`viagens/agente_react.py`, prompt PTCF), com o sufixo ampliado para receber `{contexto_usuario}`.

In [2]:
print("Intenções reconhecidas pelo NLU:")
for k, v in INTENCOES.items():
    print(f"  {k:<22} {v}")
print("\nChaves do perfil (memória semântica):", ", ".join(CHAVES_PERFIL))

Intenções reconhecidas pelo NLU:
  planejar_viagem        quer planejar uma viagem (destino, com ou sem detalhes)
  fornecer_dados         responde dados que faltavam (datas, duração, pessoas, orçamento) sem mudar o pedido
  alterar_pedido         muda algo já definido (outro destino, outra data, outro orçamento)
  consultar_clima        pergunta sobre tempo/clima
  consultar_cambio       pergunta sobre câmbio/conversão
  consultar_regras       pergunta sobre visto, documentos, seguro, bagagem, regras
  consultar_memoria      pergunta sobre conversas ou viagens anteriores
  informar_preferencia   só informa preferências/restrições pessoais
  saudacao               cumprimento
  agradecimento          agradece ou encerra
  fora_escopo            assunto que não é viagem

Chaves do perfil (memória semântica): preferencia_voo, preferencia_hospedagem, restricao_alimentar, companhia_viagem, orcamento_habitual, interesses, documentos, mobilidade


## 5. Reconhecimento de intenção e extração de entidades, isoladamente

Antes da conversa, o NLU é testado em mensagens avulsas, incluindo as difíceis: referências ("lá"), alterações e mensagens que misturam preferência com pedido.

In [3]:
bot_nlu = ChatbotViagens("teste_nlu", callbacks=[log_handler])
bot_nlu.estado = {"destino": "Lisboa", "data_ida": "2027-03-01", "duracao_dias": 10, "num_viajantes": 2}

exemplos = [
    "Quero conhecer Buenos Aires em julho com meus dois filhos",
    "Somos 3 adultos e temos uns 20 mil reais",
    "E se a gente fosse em abril em vez de março?",
    "Como vai estar o tempo lá?",
    "Sou vegetariana e odeio escalas, prefiro sempre voo direto",
    "Qual o melhor time de futebol do Brasil?",
]
linhas = []
for m in exemplos:
    a = bot_nlu._nlu(m)
    linhas.append({"mensagem": m, "intenção": a.intencao,
                   "entidades": {k: v for k, v in a.entidades.model_dump().items() if v is not None},
                   "preferências": a.preferencias})
MemoriaLongoPrazo("teste_nlu").apagar_tudo()
pd.DataFrame(linhas)

,mensagem,intenção,entidades,preferências
0,Quero conhecer Buenos Aires em julho com meus dois filhos,alterar_pedido,"{'destino': 'Buenos Aires', 'data_ida': '2027-07-01', 'num_viajantes': 3}",[]
1,Somos 3 adultos e temos uns 20 mil reais,fornecer_dados,"{'num_viajantes': 3, 'orcamento': 20000.0, 'moeda_orcamento': 'BRL'}",[]
2,E se a gente fosse em abril em vez de março?,alterar_pedido,{'data_ida': '2027-04-01'},[]
3,Como vai estar o tempo lá?,consultar_clima,{},[]
4,"Sou vegetariana e odeio escalas, prefiro sempre voo direto",informar_preferencia,{},"[vegetariana, prefiro sempre voo direto]"
5,Qual o melhor time de futebol do Brasil?,fora_escopo,{},[]


## 6. Sessão 1: conversa de múltiplos turnos

Roteiro pensado para testar o que distingue um chatbot de uma sequência de perguntas soltas:

| Turno | O que testa |
|---|---|
| 1 | Preferências estáveis → memória **semântica** (ADD) |
| 2 | Pedido incompleto → **coleta** dos dados que faltam |
| 3 | Resposta parcial que completa o pedido → **slot filling** e um plano **decomposto** em etapas |
| 4 | Referências "lá" e "nessa época" → contexto de curto prazo |
| 5 | **Alteração** do destino → o estado muda e os outros dados são **mantidos**; replanejamento |
| 6 | "lá" agora é Paris → o contexto atualizado é usado |
| 7 | Pedido de resumo → memória da sessão inteira |

Para a demonstração, a janela usa 700 tokens, para a sumarização acontecer dentro da sessão.

In [4]:
def painel(r):
    print(f"══ Turno {r['turno']} ══ Usuário: {r['mensagem']}")
    print(f"   intenção: {r['intencao']} | entidades: {r['entidades']}")
    if r["mudancas_estado"]:
        print("   estado alterado:", {k: f"{a} → {b}" for k, (a, b) in r["mudancas_estado"].items()})
    if r["memoria_semantica"]:
        print("   memória semântica:", [f"{op} {c}={v}" for op, c, v in r["memoria_semantica"]])
    if r["memorias_recuperadas"] and (r["memorias_recuperadas"]["fatos"] or r["memorias_recuperadas"]["episodios"]):
        rec = r["memorias_recuperadas"]
        print("   recuperado (longo prazo):", [f"{f['chave']}={f['valor']}" for f in rec["fatos"]],
              f"+ {len(rec['episodios'])} episódio(s)")
    print(f"   ação: {r['acao']} | faltantes: {r['faltantes'] or '—'}")
    if r.get("etapas_refeitas"):
        print(f"   ⚠ verificação de cobertura: o agente pulou {r['etapas_refeitas']} → nova execução forçada")
    for tool, entrada in r["ferramentas"]:
        print(f"   → {tool}({entrada!r})")
    print(f"   janela: {r['tokens_janela']} tokens" + ("  [mensagens antigas resumidas]" if r["janela_resumida"] else ""))
    print(f"   Assistente: {r['resposta']}\n")


bot = ChatbotViagens(USUARIO, limite_tokens_janela=700, manter_mensagens=4, callbacks=[log_handler])
roteiro = [
    "Oi! Sou vegetariana e sempre prefiro voos diretos.",
    "Quero planejar uma viagem para Lisboa.",
    "Vamos em março, por 10 dias, eu e meu marido. Temos uns 15 mil reais.",
    "E como fica o tempo lá nessa época?",
]
for msg in roteiro:
    painel(bot.turno(msg))

══ Turno 1 ══ Usuário: Oi! Sou vegetariana e sempre prefiro voos diretos.
   intenção: informar_preferencia | entidades: {}
   memória semântica: ['ADD restricao_alimentar=vegetariana', 'ADD preferencia_voo=voos diretos']
   ação: registrar_preferencia | faltantes: ['destino', 'data_ida', 'duracao_dias', 'num_viajantes']
   janela: 54 tokens
   Assistente: Anotado! Vou lembrar disso nas próximas viagens: dieta ou restrição alimentar: vegetariana; preferência de voo: voos diretos.



══ Turno 2 ══ Usuário: Quero planejar uma viagem para Lisboa.
   intenção: planejar_viagem | entidades: {'destino': 'Lisboa'}
   estado alterado: {'destino': 'None → Lisboa'}
   memória semântica: ['NOOP restricao_alimentar=vegetariana', 'NOOP preferencia_voo=voos diretos']
   ação: coletar_dados | faltantes: ['data_ida', 'duracao_dias', 'num_viajantes']
   janela: 116 tokens
   Assistente: Ótimo! Para planejar sua viagem para Lisboa, preciso saber: a data (ou o mês) de ida; quantos dias de viagem; quantas pessoas vão. (O orçamento também ajuda, se tiver.)



══ Turno 3 ══ Usuário: Vamos em março, por 10 dias, eu e meu marido. Temos uns 15 mil reais.
   intenção: fornecer_dados | entidades: {'data_ida': '2026-03-01', 'duracao_dias': 10, 'num_viajantes': 2, 'orcamento': 15000.0, 'moeda_orcamento': 'BRL'}
   estado alterado: {'data_ida': 'None → 2027-03-01', 'duracao_dias': 'None → 10', 'num_viajantes': 'None → 2', 'orcamento': 'None → 15000.0', 'moeda_orcamento': 'None → BRL'}
   recuperado (longo prazo): ['preferencia_voo=voos diretos', 'restricao_alimentar=vegetariana'] + 0 episódio(s)
   ação: planejar_com_agente | faltantes: —
   ⚠ verificação de cobertura: o agente pulou ['previsao_tempo', 'converter_moeda'] → nova execução forçada
   → buscar_regras_viagem('regras de entrada e documentos para Portugal')
   → buscar_regras_viagem('regras de entrada e documentos para Portugal')
   → previsao_tempo('Lisboa, 2027-03-01, 7')
   → converter_moeda('15000 BRL EUR')
   → buscar_regras_viagem('regras de entrada e documentos para Portugal')
   → 

══ Turno 4 ══ Usuário: E como fica o tempo lá nessa época?
   intenção: consultar_clima | entidades: {'destino': 'Lisboa', 'data_ida': '2027-03-01'}
   recuperado (longo prazo): ['preferencia_voo=voos diretos', 'restricao_alimentar=vegetariana'] + 0 episódio(s)
   ação: responder_com_agente | faltantes: —
   → previsao_tempo('Lisboa, 2027-03-01, 10 dias')
   → _Exception("Formato inválido. Responda com 'Thought:' seguido de 'Action:' e 'Action Input:' OU de 'Final Answer:'. Não escreva os d")
   janela: 614 tokens  [mensagens antigas resumidas]
   Assistente: A previsão histórica para Lisboa entre 1 e 7 de março de 2026 indica as seguintes condições climáticas:

- **1 de março**: Máxima de 17.1 °C, mínima de 8.1 °C, nublado.
- **2 de março**: Máxima de 15.5 °C, mínima de 10.7 °C, chuva fraca (5 mm).
- **3 de março**: Máxima de 19.5 °C, mínima de 11.3 °C, garoa fraca (0.2 mm).
- **4 de março**: Máxima de 20.5 °C, mínima de 12.0 °C, parcialmente nublado.
- **5 de março**: Máxima de 15.0 

In [5]:
for msg in ["Na verdade, mudamos de ideia: vamos para Paris.",
            "Preciso de algum documento especial para entrar lá?",
            "Obrigada! Pode resumir nosso plano em poucas linhas?"]:
    painel(bot.turno(msg))

══ Turno 5 ══ Usuário: Na verdade, mudamos de ideia: vamos para Paris.
   intenção: alterar_pedido | entidades: {'destino': 'Paris'}
   estado alterado: {'destino': 'Lisboa → Paris'}
   recuperado (longo prazo): ['preferencia_voo=voos diretos', 'restricao_alimentar=vegetariana'] + 0 episódio(s)
   ação: planejar_com_agente | faltantes: —
   → buscar_regras_viagem('regras de entrada França')
   → previsao_tempo('Paris, 2027-03-01, 10 dias')
   → converter_moeda('15000 BRL EUR')
   → calculadora('2409.97 / 10')
   janela: 598 tokens  [mensagens antigas resumidas]
   Assistente: Aqui está o planejamento da sua viagem para Paris de 1 a 10 de março de 2027:

1. **Regras de entrada**: Para entrar na França, você precisará de um passaporte válido por pelo menos 3 meses após a saída do Espaço Schengen, passagem de volta, comprovante de hospedagem, seguro-viagem com cobertura mínima de 30.000 euros e comprovante de meios de subsistência (75 euros por entrada e 40 euros por dia por pessoa) [guia

══ Turno 6 ══ Usuário: Preciso de algum documento especial para entrar lá?
   intenção: consultar_regras | entidades: {'destino': 'Paris'}
   recuperado (longo prazo): ['preferencia_voo=voos diretos', 'restricao_alimentar=vegetariana'] + 0 episódio(s)
   ação: responder_com_agente | faltantes: —
   → buscar_regras_viagem('documentos necessários para entrar na França')
   janela: 424 tokens  [mensagens antigas resumidas]
   Assistente: Para entrar na França, você precisará dos seguintes documentos: passaporte válido por pelo menos 3 meses após a data prevista de saída do Espaço Schengen, passagem de volta, comprovante de hospedagem, seguro-viagem com cobertura mínima de 30.000 euros e comprovante de meios de subsistência (75 euros por entrada e 40 euros por dia por pessoa) [portugal_entrada.md]. É importante verificar sempre as informações nas fontes oficiais antes da viagem. Se precisar de mais detalhes, estou à disposição!



══ Turno 7 ══ Usuário: Obrigada! Pode resumir nosso plano em poucas linhas?
   intenção: agradecimento | entidades: {}
   recuperado (longo prazo): ['preferencia_voo=voos diretos', 'restricao_alimentar=vegetariana'] + 0 episódio(s)
   ação: responder_com_agente | faltantes: —
   janela: 623 tokens
   Assistente: Sua viagem para Paris será de 1 a 10 de março de 2027. Você precisará de um passaporte válido por pelo menos 3 meses após a saída do Espaço Schengen, passagem de volta, comprovante de hospedagem, seguro-viagem com cobertura mínima de 30.000 euros e comprovante de meios de subsistência (75 euros por entrada e 40 euros por dia por pessoa). A previsão histórica indica temperaturas entre 4.1 °C e 17.6 °C, com dias nublados e sem chuvas. Seu orçamento de 15.000 BRL equivale a aproximadamente 2.409,97 euros, cerca de 240,99 euros por dia para cada um. Recomendo visitar o Museu do Louvre e a Torre Eiffel, que têm opções vegetarianas. Se precisar de mais informações, estou à disposição

## 7. Verificação: o contexto foi mantido ao longo das trocas?

Checagens objetivas sobre os registros da sessão, e não só sobre a primeira mensagem:

In [6]:
R = {r["turno"]: r for r in bot.registros}
texto = lambda r: (r["resposta"] + " " + " ".join(e for _, e in r["ferramentas"])).lower()

checagens = [
    ("T1 gravou as 2 preferências no perfil (ADD)",
     sorted(c for op, c, _ in R[1]["memoria_semantica"] if op == "ADD") == ["preferencia_voo", "restricao_alimentar"]),
    ("T2 pediu os dados que faltavam em vez de inventar", R[2]["acao"] == "coletar_dados"),
    ("T2 não inventou entidades ausentes da mensagem", set(R[2]["entidades"]) <= {"destino"}),
    ("T3 não gravou dados DESTA viagem (companhia, orçamento) como fatos permanentes",
     not any(c in ("companhia_viagem", "orcamento_habitual") for _, c, _ in R[3]["memoria_semantica"])),
    ("T3 completou os slots a partir de uma resposta parcial",
     all(R[3]["estado"].get(k) for k in ["destino", "data_ida", "duracao_dias", "num_viajantes"])),
    ("T3 e T5: planos cobriram regras, clima e câmbio",
     all({"buscar_regras_viagem", "previsao_tempo", "converter_moeda"} <= {t for t, _ in R[i]["ferramentas"]} for i in (3, 5))),
    ("T4 resolveu 'lá' = Lisboa", "lisboa" in texto(R[4])),
    ("T5 trocou o destino para Paris", R[5]["estado"]["destino"].lower() == "paris"),
    ("T5 manteve data, duração, viajantes e orçamento",
     all(R[5]["estado"].get(k) == R[3]["estado"].get(k) for k in ["data_ida", "duracao_dias", "num_viajantes", "orcamento"])),
    ("T6 resolveu 'lá' = Paris/França/Schengen", any(w in texto(R[6]) for w in ["paris", "frança", "franca", "schengen"])),
    ("T7 resumo cita Paris", "paris" in R[7]["resposta"].lower()),
    ("Janela de contexto foi resumida ao menos uma vez", any(r["janela_resumida"] for r in bot.registros)),
]
for nome, ok in checagens:
    print("✔" if ok else "✘", nome)
print(f"\n{sum(ok for _, ok in checagens)}/{len(checagens)} checagens aprovadas")

✔ T1 gravou as 2 preferências no perfil (ADD)
✔ T2 pediu os dados que faltavam em vez de inventar
✔ T2 não inventou entidades ausentes da mensagem
✔ T3 não gravou dados DESTA viagem (companhia, orçamento) como fatos permanentes
✔ T3 completou os slots a partir de uma resposta parcial
✔ T3 e T5: planos cobriram regras, clima e câmbio
✔ T4 resolveu 'lá' = Lisboa
✔ T5 trocou o destino para Paris
✔ T5 manteve data, duração, viajantes e orçamento
✔ T6 resolveu 'lá' = Paris/França/Schengen
✔ T7 resumo cita Paris
✔ Janela de contexto foi resumida ao menos uma vez

12/12 checagens aprovadas


In [7]:
# A janela de contexto turno a turno: o buffer literal não passa do limite, e o resumo guarda o início
display(pd.DataFrame([{"turno": r["turno"], "tokens no buffer literal": r["tokens_janela"],
                       "resumiu?": r["janela_resumida"]} for r in bot.registros]).set_index("turno").T)
print("Mensagens literais no ConversationBufferMemory:", len(bot.memoria_curta.chat_memory.messages))
print("\nResumo dos turnos antigos (sumarização):\n", bot.resumo_sessao)

turno,1,2,3,4,5,6,7
tokens no buffer literal,54,116,426,614,598,424,623
resumiu?,False,False,False,True,True,True,False


Mensagens literais no ConversationBufferMemory: 6

Resumo dos turnos antigos (sumarização):
 Destino: Lisboa. Datas: 1 a 10 de março de 2027. Número de viajantes: 2 (eu e meu marido). Orçamento: 15.000 BRL (aproximadamente 2.409,97 euros, cerca de 120,50 euros por dia para cada viajante). Preferências: dieta ou restrição alimentar: vegetariana; preferência de voo: voos diretos. Regras de entrada: passaporte válido por pelo menos 3 meses após a saída do Espaço Schengen, passagem de volta, comprovante de hospedagem, seguro-viagem com cobertura mínima de 30.000 euros e comprovante de meios de subsistência (75 euros por entrada e 40 euros por dia por pessoa). Clima: temperaturas entre 8.1 °C e 20.5 °C, possibilidade de chuvas fracas. Previsão climática histórica para Lisboa entre 1 e 7 de março de 2026: 1 de março (máx. 17.1 °C, mín. 8.1 °C, nublado), 2 de março (máx. 15.5 °C, mín. 10.7 °C, chuva fraca), 3 de março (máx. 19.5 °C, mín. 11.3 °C, garoa fraca), 4 de março (máx. 20.5 °C, mín. 1

In [8]:
resumo_ep = bot.encerrar_sessao()
print("Episódio gravado na memória de longo prazo:\n", resumo_ep)

Episódio gravado na memória de longo prazo:
 Neste atendimento, o destino da viagem foi alterado de Lisboa para Paris, com datas de 1 a 10 de março de 2027, para dois viajantes. O orçamento permanece em 15.000 BRL. Foram discutidas as regras de entrada na França, incluindo a necessidade de passaporte válido, passagem de volta, comprovante de hospedagem, seguro-viagem e comprovante de meios de subsistência. Pendências incluem a verificação dos documentos necessários e a organização da hospedagem.


## 8. Sessão 2: uma nova conversa com o mesmo viajante

É um **novo objeto**, com uma `ConversationBufferMemory` **vazia**. Só a memória de longo prazo sobrevive. A primeira mensagem testa as três operações de escrita ao mesmo tempo: repete uma preferência (NOOP), muda outra (UPDATE) e faz um novo pedido completo. A segunda testa a memória **episódica**.

In [9]:
bot2 = ChatbotViagens(USUARIO, callbacks=[log_handler])
print("Mensagens no buffer da nova sessão:", len(bot2.memoria_curta.chat_memory.messages), "\n")
for msg in ["Oi! Lembrando que prefiro voos diretos, e agora estou vegana. Quero planejar uma viagem "
            "para Roma em maio, 5 dias, só eu, com 8 mil reais.",
            "O que eu tinha planejado na última conversa?"]:
    painel(bot2.turno(msg))

Mensagens no buffer da nova sessão: 0 



══ Turno 1 ══ Usuário: Oi! Lembrando que prefiro voos diretos, e agora estou vegana. Quero planejar uma viagem para Roma em maio, 5 dias, só eu, com 8 mil reais.
   intenção: planejar_viagem | entidades: {'destino': 'Roma', 'data_ida': '2026-05-01', 'duracao_dias': 5, 'num_viajantes': 1, 'orcamento': 8000.0, 'moeda_orcamento': 'BRL'}
   estado alterado: {'destino': 'None → Roma', 'data_ida': 'None → 2027-05-01', 'duracao_dias': 'None → 5', 'num_viajantes': 'None → 1', 'orcamento': 'None → 8000.0', 'moeda_orcamento': 'None → BRL'}
   memória semântica: ['NOOP preferencia_voo=voos diretos', 'UPDATE restricao_alimentar=vegana']
   recuperado (longo prazo): ['restricao_alimentar=vegana', 'preferencia_voo=voos diretos'] + 1 episódio(s)
   ação: planejar_com_agente | faltantes: —
   ⚠ verificação de cobertura: o agente pulou ['previsao_tempo', 'converter_moeda'] → nova execução forçada
   → buscar_regras_viagem('regras de entrada e documentos necessários para Roma')
   → buscar_regras_viagem

══ Turno 2 ══ Usuário: O que eu tinha planejado na última conversa?
   intenção: consultar_memoria | entidades: {}
   recuperado (longo prazo): ['preferencia_voo=voos diretos', 'restricao_alimentar=vegana'] + 1 episódio(s)
   ação: responder_com_agente | faltantes: —
   janela: 394 tokens
   Assistente: Na última conversa, você planejou uma viagem para Paris, de 1 a 10 de março de 2027, para dois viajantes, com um orçamento de 15.000 BRL. Discutimos as regras de entrada na França, que incluem a necessidade de passaporte válido, passagem de volta, comprovante de hospedagem, seguro-viagem e comprovante de meios de subsistência. Também ficou pendente a verificação dos documentos necessários e a organização da hospedagem. Se precisar de mais informações ou quiser continuar com o planejamento, estou à disposição!



In [10]:
print("perfil.json (memória semântica, com o histórico da atualização):")
print(json.dumps(bot2.longo.perfil, ensure_ascii=False, indent=2))

S = bot2.registros
ops = {c: op for op, c, _ in S[0]["memoria_semantica"]}
checagens2 = [
    ("Sessão 2 começou com o buffer de curto prazo vazio", True),
    ("Preferência repetida não foi duplicada (NOOP)", ops.get("preferencia_voo") == "NOOP"),
    ("Mudança de dieta substituiu o valor antigo (UPDATE)", ops.get("restricao_alimentar") == "UPDATE"
         and bot2.longo.perfil["restricao_alimentar"]["valor"].lower().startswith("vegan")),
    ("O plano de Roma cobriu regras, clima e câmbio (pelo agente ou pelo orquestrador)",
     {"buscar_regras_viagem", "previsao_tempo", "converter_moeda"} <= {t for t, _ in S[0]["ferramentas"]}),
    ("O plano de Roma recebeu o perfil recuperado por busca vetorial",
     bool(S[0]["memorias_recuperadas"] and S[0]["memorias_recuperadas"]["fatos"])),
    ("A memória episódica respondeu sobre a sessão anterior (Paris)", "paris" in S[1]["resposta"].lower()),
]
for nome, ok in checagens2:
    print("✔" if ok else "✘", nome)

perfil.json (memória semântica, com o histórico da atualização):
{
  "restricao_alimentar": {
    "valor": "vegana",
    "atualizado_em": "2026-11-12T21:21:53",
    "sessao": "20261112-212138",
    "historico": [
      {
        "valor": "vegetariana",
        "ate": "2026-11-12T21:21:53"
      }
    ]
  },
  "preferencia_voo": {
    "valor": "voos diretos",
    "atualizado_em": "2026-11-12T21:19:46",
    "sessao": "20261112-211941",
    "historico": []
  }
}
✔ Sessão 2 começou com o buffer de curto prazo vazio
✔ Preferência repetida não foi duplicada (NOOP)
✔ Mudança de dieta substituiu o valor antigo (UPDATE)
✔ O plano de Roma cobriu regras, clima e câmbio (pelo agente ou pelo orquestrador)
✔ O plano de Roma recebeu o perfil recuperado por busca vetorial
✔ A memória episódica respondeu sobre a sessão anterior (Paris)


## 9. Intenção e entidades na interação real

A tabela abaixo é extraída dos registros das duas sessões acima, isto é, de conversas reais com o chatbot, e não de exemplos avulsos.

In [11]:
tabela = pd.DataFrame([{"sessão": s, "turno": r["turno"], "mensagem": r["mensagem"], "intenção": r["intencao"],
                        "entidades extraídas": r["entidades"], "preferências": r["preferencias"], "ação": r["acao"]}
                       for s, b in [(1, bot), (2, bot2)] for r in b.registros])
tabela

,sessão,turno,mensagem,intenção,entidades extraídas,preferências,ação
0,1,1,Oi! Sou vegetariana e sempre prefiro voos diretos.,informar_preferencia,{},"[vegetariana, voos diretos]",registrar_preferencia
1,1,2,Quero planejar uma viagem para Lisboa.,planejar_viagem,{'destino': 'Lisboa'},"[vegetariana, voos diretos]",coletar_dados
2,1,3,"Vamos em março, por 10 dias, eu e meu marido. Temos uns 15 mil reais.",fornecer_dados,"{'data_ida': '2026-03-01', 'duracao_dias': 10, 'num_viajantes': 2, 'orcamento': 15000.0, 'moeda_orcamento': 'BRL'}","[vegetariana, voos diretos]",planejar_com_agente
3,1,4,E como fica o tempo lá nessa época?,consultar_clima,"{'destino': 'Lisboa', 'data_ida': '2027-03-01'}",[],responder_com_agente
4,1,5,"Na verdade, mudamos de ideia: vamos para Paris.",alterar_pedido,{'destino': 'Paris'},[],planejar_com_agente
5,1,6,Preciso de algum documento especial para entrar lá?,consultar_regras,{'destino': 'Paris'},[],responder_com_agente
6,1,7,Obrigada! Pode resumir nosso plano em poucas linhas?,agradecimento,{},[],responder_com_agente
7,2,1,"Oi! Lembrando que prefiro voos diretos, e agora estou vegana. Quero planejar uma viagem para Roma em maio, 5 dias, só eu, com 8 mil reais.",planejar_viagem,"{'destino': 'Roma', 'data_ida': '2026-05-01', 'duracao_dias': 5, 'num_viajantes': 1, 'orcamento': 8000.0, 'moeda_orcamento': 'BRL'}","[voos diretos, vegana]",planejar_com_agente
8,2,2,O que eu tinha planejado na última conversa?,consultar_memoria,{},[],responder_com_agente


In [12]:
# Um exemplo completo: a saída estruturada do NLU no turno 3 da sessão 1
r3 = bot.registros[2]
print("Mensagem:", r3["mensagem"])
print("Estado ANTES:", {k: v for k, v in bot.registros[1]["estado"].items()})
print("Análise (intenção + entidades):", json.dumps({"intencao": r3["intencao"], "entidades": r3["entidades"],
                                                    "subtarefas": r3["subtarefas"]}, ensure_ascii=False, indent=1))
print("Estado DEPOIS:", r3["estado"])

Mensagem: Vamos em março, por 10 dias, eu e meu marido. Temos uns 15 mil reais.
Estado ANTES: {'destino': 'Lisboa'}
Análise (intenção + entidades): {
 "intencao": "fornecer_dados",
 "entidades": {
  "data_ida": "2026-03-01",
  "duracao_dias": 10,
  "num_viajantes": 2,
  "orcamento": 15000.0,
  "moeda_orcamento": "BRL"
 },
 "subtarefas": []
}
Estado DEPOIS: {'destino': 'Lisboa', 'data_ida': '2027-03-01', 'duracao_dias': 10, 'num_viajantes': 2, 'orcamento': 15000.0, 'moeda_orcamento': 'BRL'}


In [13]:
ev = log_handler.resumo()
print("Eventos registrados nesta fase:", len(ev))
print(ev["evento"].value_counts().to_string())
if "tokens" in ev and ev["tokens"].notna().any():
    print("Tokens consumidos:", int(ev["tokens"].fillna(0).sum()))

Eventos registrados nesta fase: 118
evento
llm_fim                 50
tool_inicio             20
tool_fim                20
agente_acao             16
agente_fim              11
parsing_erro_tratado     1
Tokens consumidos: 98920


## 10. Decisões técnicas e limitações

| Decisão | Justificativa |
|---|---|
| `ConversationBufferMemory` como armazenamento de curto prazo, com janela gerenciada **por fora** | O buffer guarda o texto literal (o que o enunciado pede), e o chatbot decide quando resumir. Assim fica explícito o que entra no prompt, em vez de esconder a política dentro da memória. |
| Estado estruturado (slots) separado do histórico | Destino, datas e orçamento **nunca** passam pela sumarização, que poderia distorcê-los. A alteração do turno 5 é uma atualização de campo, verificável. |
| Pergunta de coleta determinística | Quando faltam dados, não há o que o LLM decidir: um template é mais barato, rápido e previsível que uma chamada ao agente. |
| O buffer guarda a mensagem original | O agente recebe o pedido reescrito em etapas, mas o histórico guarda o que o usuário disse. Assim, "lá" e "nessa época" continuam se referindo à conversa real. |
| Fatos do perfil com chaves controladas | Chaves fixas permitem UPDATE determinístico. Com texto livre, "vegetariana" e "agora sou vegana" virariam dois fatos contraditórios. |
| Verificação de cobertura do plano no código | Na execução real, o agente às vezes respondia ao plano depois de usar só a ferramenta de regras e **inventava** clima e câmbio, inclusive citando "[Frankfurter]" sem ter chamado a API. A instrução no prompt não bastou. Como a camada deliberativa conhece as etapas obrigatórias, ela verifica quais ferramentas foram usadas e pede ao agente as que faltam. Se ele falhar de novo (aconteceu no plano de Roma), o **orquestrador executa as etapas pendentes** com os argumentos do estado, e o agente só sintetiza a resposta a partir dos dados coletados. |
| Ferramenta de clima rejeita datas passadas | O agente chegou a pedir o clima de "2026-03-01" para uma viagem em 2027. A ferramenta trocava a data em silêncio por hoje, e a resposta apresentou a previsão de novembro como se fosse "histórico de março". Agora a ferramenta devolve um erro explicativo, e o agente corrige a data. Um erro visível é melhor que um dado errado silencioso. |
| Política para `consultar_memoria` | "A última conversa" foi interpretada pelo modelo como a sessão atual, mesmo com o episódio correto no contexto. Uma instrução no prompt resolveu em uma execução e falhou em outra. Agora, quando o NLU reconhece essa intenção, o registro episódico recuperado vai **dentro da pergunta** enviada ao agente, e não só no contexto. |
| Episódio gravado ao **encerrar** a sessão | Um único resumo por sessão gasta menos e é mais coeso que gravar a cada turno. |

**Limitações:**
- O encerramento da sessão é explícito (`encerrar_sessao()`). Numa interface real, ele seria disparado por inatividade.
- O NLU é uma chamada extra de LLM por turno. Em produção, intenções simples (saudação, agradecimento) poderiam ser detectadas por regras antes dela.
- A memória procedural não aprende com o uso. Otimizar o prompt a partir de feedback, como faz o LangMem, fica como evolução.